# LLM as Judge: Evaluating BDD Test Case Quality

This notebook uses an LLM to evaluate the three types of BDD tests generated in `test_case_generation.ipynb`:
1. **Simple** – generated from app name only  
2. **With context** – generated from app name + app context  
3. **From graph** – generated from app screen graph (nodes/edges)

We define a rubric and evaluation criteria, then call the LLM to score and compare the test sets.

In [26]:
# Configuration: app context and API key
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
# OpenAI API key (from .env OPENAI_API_KEY or environment variable)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

# App context: same as in test_case_generation.ipynb — used by the judge to assess relevance
APP_CONTEXT = """
Book travel accommodation & save! Compare hotel deals, apartments & hostels

Download the highest rated travel app and join thousands of people finding hotel, motel and holiday home travel deals. Find a stay for your holiday, weekend getaway or business trip, anywhere in the world!

Search for Hotels, Motels & Holiday Homes

* 1,5+ Million properties: hotels, motels, vacation rentals and more
* Search by city, attraction, landmark or hotel name with just one tap
* Filter by price, review score, WiFi quality and other things important to you
* Find your last-minute stay or book far in advance

Find a Great Travel Deal

* Daily deals for every budget
* Free cancellation on most rooms
* No booking or credit card fees
* Browse 135+ Million verified guest reviews

Manage your booking on the go

* Get paperless confirmation 
* Make changes whenever, wherever
* 24/7 customer service in more than 40 languages
* Contact your host, view check-in times and leave behind reviews
"""

# Paths to the three generated test case files (from test_case_generation.ipynb)
APP_NAME = "booking.com"  # must match the prefix used when generating test cases
OUTPUT_DIR_SIMPLE = Path("test_cases_simple")
OUTPUT_DIR_CONTEXT = Path("test_cases_with_context")
OUTPUT_DIR_GRAPH = Path("test_cases_from_graph")

# Model used for LLM judge evaluation (e.g. gpt-4o-mini, gpt-4o)
MODEL_NAME = os.environ.get("OPENAI_MODEL")

## Rubric and evaluation criteria

The LLM judge scores each test set on **six weighted criteria** (1–5 per criterion).  
The **weighted overall score** = Σ(score × weight), normalised to 1–5.

| # | Criterion | Weight | What a **1** looks like | What a **3** looks like | What a **5** looks like |
|---|-----------|--------|------------------------|------------------------|------------------------|
| 1 | **BDD Structure** | 25 % | Missing Given/When/Then; mixed steps; invalid Gherkin syntax | Some scenarios well-formed; occasional missing keywords or combined steps | All scenarios follow strict Gherkin; atomic, single-action steps; correct keyword use throughout |
| 2 | **Domain Relevance** | 20 % | Completely generic; could apply to any app | Mentions the app domain but lacks specific vocabulary, realistic data, or app-specific flows | Uses domain terms, realistic data values (e.g. prices, dates), and covers app-specific features |
| 3 | **Coverage** | 20 % | Only one flow (e.g. happy path only, or a single screen) | Core flows present but no edge cases or error states | Covers happy paths, navigation, edge cases (empty results, network errors, invalid input), and key user journeys |
| 4 | **Clarity & Actionability** | 15 % | Vague steps like "user interacts with app"; nothing an automator could implement | Most steps are clear but some reference abstract UI ("the button") | Every step names a concrete UI element, action, and expected result; unambiguous and directly automatable |
| 5 | **Scenario Quality** | 15 % | Scenarios are incomplete, duplicate, or share implicit state; no meaningful titles | Most scenarios are self-contained and titled, but some overlap or have trivial steps | Every scenario is independent, meaningfully titled, uses realistic test data, and has no redundant or trivial steps |
| 6 | **Traceability** | 5 % | No visible link to the source (name / context / graph) | Weakly linked—some features align but others seem invented | Clearly derived from source: *Simple* fits app name; *Context* reflects all provided app features; *Graph* maps every node/edge |

**Weighted overall score** = (bdd×0.25 + domain×0.20 + coverage×0.20 + clarity×0.15 + quality×0.15 + trace×0.05), rounded to one decimal place (1.0–5.0).

The judge returns a `justification` object with one sentence per criterion explaining each score.

In [27]:
# Load the three BDD test sets generated by test_case_generation.ipynb
import json

def load_test_cases(dir_path: Path, filename: str) -> list[dict]:
    path = dir_path / filename
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)

name_safe = APP_NAME.replace(" ", "_")
simple = load_test_cases(OUTPUT_DIR_SIMPLE, f"{name_safe}_test_cases.json")
with_context = load_test_cases(OUTPUT_DIR_CONTEXT, f"{name_safe}_test_cases.json")
from_graph = load_test_cases(OUTPUT_DIR_GRAPH, f"{name_safe}_graph_test_cases.json")

print(f"Simple (app name only):     {len(simple)} feature(s)")
print(f"With context:               {len(with_context)} feature(s)")
print(f"From graph:                 {len(from_graph)} feature(s)")

Simple (app name only):     5 feature(s)
With context:               8 feature(s)
From graph:                 10 feature(s)


In [28]:
# LLM judge: evaluation prompt and criteria
from openai import OpenAI

if not OPENAI_API_KEY or OPENAI_API_KEY == "your-api-key-here":
    raise ValueError("Set OPENAI_API_KEY in the .env file or in your environment.")

client = OpenAI(api_key=OPENAI_API_KEY)

EVALUATION_CRITERIA = """
You are an expert QA engineer judging BDD (Given/When/Then) test case quality.

TEST CASE FORMAT
================
The test cases are provided as a JSON array of plain strings, one scenario per string,
with steps separated by newlines. They do NOT include Feature: or Scenario: wrappers —
that is intentional and must NOT be penalised. A well-formed scenario looks like:
  "Given <precondition>\nWhen <action>\nThen <expected result>"
Evaluate only the content and quality of the steps themselves.

CRITERIA AND WEIGHTS
====================

1. bdd_structure (weight 0.25)
   1 – Given/When/Then keywords missing or used incorrectly; steps are not distinguishable.
   3 – Keywords present in most scenarios but some steps combine multiple actions or misuse And/But.
   5 – Every scenario has exactly one Given, one When, one Then (with And only for natural continuations); all steps are atomic.

2. domain_relevance (weight 0.20)
   1 – Completely generic; could apply to any app.
   3 – Mentions the domain but lacks specific vocabulary, realistic data, or app-specific flows.
   5 – Uses domain terms and realistic data values; covers app-specific features throughout.

3. coverage (weight 0.20)
   1 – Only one flow (e.g. happy path only, or a single screen).
   3 – Core flows present but no edge cases or error states.
   5 – Covers happy paths, navigation, edge cases (empty results, errors, invalid input), and key user journeys.

4. clarity_actionability (weight 0.15)
   1 – Vague steps ("user interacts with app"); nothing an automator could implement.
   3 – Most steps clear but some reference abstract UI elements ("the button").
   5 – Every step names a concrete UI element, action, and expected result; directly automatable.

5. scenario_quality (weight 0.15)
   1 – Scenarios are incomplete, duplicate, or share implicit state; no meaningful titles.
   3 – Most scenarios are self-contained and titled, but some overlap or have trivial steps.
   5 – Every scenario is independent, meaningfully titled, uses realistic test data, no redundant steps.

6. traceability (weight 0.05)
   1 – No visible link to the source (app name / context / graph).
   3 – Weakly linked—some features align but others seem invented.
   5 – Clearly derived from source: Simple fits app name; Context reflects all provided features; Graph maps every screen node/edge.

SCORING
=======
Compute overall = round(bdd*0.25 + domain*0.20 + coverage*0.20 + clarity*0.15 + quality*0.15 + trace*0.05, 1).
Scores are floats to one decimal place.

OUTPUT FORMAT
=============
Respond with valid JSON only — no markdown, no extra text — in this exact shape:

{
  "evaluations": [
    {
      "test_type": "simple",
      "scores": {
        "bdd_structure": N,
        "domain_relevance": N,
        "coverage": N,
        "clarity_actionability": N,
        "scenario_quality": N,
        "traceability": N
      },
      "overall": N,
      "justification": {
        "bdd_structure": "One sentence explaining why this score was given.",
        "domain_relevance": "One sentence explaining why this score was given.",
        "coverage": "One sentence explaining why this score was given.",
        "clarity_actionability": "One sentence explaining why this score was given.",
        "scenario_quality": "One sentence explaining why this score was given.",
        "traceability": "One sentence explaining why this score was given."
      }
    },
    { "test_type": "with_context", "scores": {}, "overall": N, "justification": {} },
    { "test_type": "from_graph",   "scores": {}, "overall": N, "justification": {} }
  ]
}
"""

In [29]:
# Run the LLM judge
import re

def evaluate_bdd_tests(
    app_context: str,
    simple_tests: list,
    context_tests: list,
    graph_tests: list,
    model: str = None,
) -> dict:
    """Call LLM to evaluate the three BDD test sets; returns parsed evaluation JSON."""
    if model is None:
        model = MODEL_NAME
    user_content = f"""App context (what the app is):
{app_context.strip()}

---

Test set 1 — SIMPLE (generated from app name only):
{json.dumps(simple_tests, indent=2)}

---

Test set 2 — WITH CONTEXT (generated from app name + app context):
{json.dumps(context_tests, indent=2)}

---

Test set 3 — FROM GRAPH (generated from app screen graph):
{json.dumps(graph_tests, indent=2)}

---

Evaluate each of the three test sets using the six criteria. Return only the JSON object (no markdown)."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": EVALUATION_CRITERIA},
            {"role": "user", "content": user_content},
        ],
        temperature=0.2,
    )
    text = response.choices[0].message.content.strip()
    if "```json" in text:
        text = re.search(r"```(?:json)?\s*([\s\S]*?)```", text).group(1).strip()
    elif "```" in text:
        text = re.sub(r"^```\w*\s*", "", text).strip()
        text = re.sub(r"\s*```$", "", text).strip()
    return json.loads(text)

In [30]:
# Evaluate the three BDD test sets
result = evaluate_bdd_tests(
    app_context=APP_CONTEXT,
    simple_tests=simple,
    context_tests=with_context,
    graph_tests=from_graph,
    model=MODEL_NAME,
)

# Pretty-print evaluations
for ev in result["evaluations"]:
    print(f"\n=== {ev['test_type'].upper().replace('_', ' ')} ===")
    print(f"Overall: {ev['overall']}/5\n")
    for criterion, score in ev["scores"].items():
        justification = ev["justification"][criterion]
        print(f"  {criterion:<25} {score}  —  {justification}")



=== SIMPLE ===
Overall: 4.3/5

  bdd_structure             5  —  All scenarios use correct Given/When/Then structure with atomic steps and no misuse of And/But.
  domain_relevance          4  —  Scenarios reference travel-specific terms and actions but lack some of the richer vocabulary and features from the app context.
  coverage                  4  —  Happy paths, booking, search, login, and cancellation are covered, but there are few edge cases or error flows.
  clarity_actionability     5  —  Each step specifies concrete UI elements and actions, making them directly automatable.
  scenario_quality          4  —  Scenarios are independent, realistic, and use plausible data, but some flows are basic and could be more detailed.
  traceability              3  —  Scenarios align with the app name and basic flows but do not reflect the full app context or screen graph.

=== WITH CONTEXT ===
Overall: 4.7/5

  bdd_structure             5  —  All scenarios follow the Given/When/Then forma

In [31]:
# Full evaluation result (for inspection or downstream use)
result

{'evaluations': [{'test_type': 'simple',
   'scores': {'bdd_structure': 5,
    'domain_relevance': 4,
    'coverage': 4,
    'clarity_actionability': 5,
    'scenario_quality': 4,
    'traceability': 3},
   'overall': 4.3,
   'justification': {'bdd_structure': 'All scenarios use correct Given/When/Then structure with atomic steps and no misuse of And/But.',
    'domain_relevance': 'Scenarios reference travel-specific terms and actions but lack some of the richer vocabulary and features from the app context.',
    'coverage': 'Happy paths, booking, search, login, and cancellation are covered, but there are few edge cases or error flows.',
    'clarity_actionability': 'Each step specifies concrete UI elements and actions, making them directly automatable.',
    'scenario_quality': 'Scenarios are independent, realistic, and use plausible data, but some flows are basic and could be more detailed.',
    'traceability': 'Scenarios align with the app name and basic flows but do not reflect th